# Prédiction du milieu de la fourchette salariale

Le projet ne doit pas rester seulement descriptif. Nous testons donc si le métier, le pays, l'expérience, le mode de travail, le type de contrat et les compétences permettent de **prédire le salaire d'une offre** (milieu de la fourchette minimum–maximum, en USD ; ce n'est pas un salaire versé). Les bornes salariales, l'identifiant et le nombre de candidats sont exclus des variables explicatives, sinon le modèle « tricherait ».

**Protocole.** Découpage aléatoire reproductible : 80 % des offres pour l'entraînement, 20 % pour le test, sans offre commune. L'encodage des catégories est ajusté uniquement sur l'entraînement et aucun réglage n'est choisi à partir du test. Les compétences sont codées en présence/absence (1 ou 0).

**Trois modèles sont comparés :**
1. *Référence moyenne* : prédit toujours le salaire moyen de l'entraînement (le minimum à battre) ;
2. *Moyenne par métier* : prédit le salaire moyen du métier de l'offre, calculé sur l'entraînement ;
3. *Régression Ridge* : utilise toutes les variables (métier, pays, expérience, mode, contrat, 26 compétences).

Comparer Ridge à la moyenne par métier permet de savoir si les autres variables apportent une information en plus du métier.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
project_root = Path.cwd() if (Path.cwd() / "data/archive").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.salary_prediction import evaluate_salary_model
data_dir = project_root / "data/archive"
offres = pd.read_csv(data_dir / "fact_job_postings.csv")
liens = pd.read_csv(data_dir / "bridge_job_skills.csv")
scores, predictions = evaluate_salary_model(offres, liens)
display(scores.round(3))

,MAE_USD,RMSE_USD,R2,n_train,n_test
modele,,,,,
Reference moyenne,31452.332,36998.196,-0.000,44000,11000
Moyenne par metier,17870.604,21318.457,0.668,44000,11000
Regression Ridge,17872.053,21325.897,0.668,44000,11000


In [3]:
output = project_root / "data/processed"
output.mkdir(exist_ok=True)
scores.to_csv(output / "salary_model_metrics.csv")
predictions.to_csv(output / "salary_test_predictions.csv", index=False)
display(predictions.head(10).round(2))

mae = scores["MAE_USD"]
print(f"Réduction de MAE de Ridge par rapport à la moyenne globale : {100 * (1 - mae['Regression Ridge'] / mae['Reference moyenne']):.2f} %")
print(f"Réduction de MAE de Ridge par rapport à la moyenne par métier : {100 * (1 - mae['Regression Ridge'] / mae['Moyenne par metier']):.2f} %")

,job_id,salaire_cible_usd,Reference moyenne,Moyenne par metier,Regression Ridge
0,JOB-34192,125157.5,121996.25,125450.04,125475.08
1,JOB-23680,93159.5,121996.25,77597.53,77267.70
2,JOB-44685,126725.5,121996.25,124693.29,125516.67
3,JOB-11336,146559.0,121996.25,124693.29,124208.94
4,JOB-22692,184421.5,121996.25,147485.24,147201.87
5,JOB-32405,115737.5,121996.25,147819.59,147811.13
6,JOB-53453,103650.0,121996.25,125450.04,125369.84
7,JOB-50290,163217.0,121996.25,147030.34,148091.36
8,JOB-587,109526.0,121996.25,147819.59,147366.73
9,JOB-23901,96602.0,121996.25,125450.04,125540.53


Réduction de MAE de Ridge par rapport à la moyenne globale : 43.18 %
Réduction de MAE de Ridge par rapport à la moyenne par métier : -0.01 %


## Interprétation

- Ridge réduit l'erreur moyenne (MAE) d'environ **43 %** par rapport à la moyenne globale : 17 872 USD contre 31 452 USD, avec un R² de 0,668.
- Mais la simple **moyenne par métier** obtient exactement le même score (MAE 17 871 USD, R² 0,668). Ajouter le pays, l'expérience, le mode de travail, le contrat et les 26 compétences n'apporte **aucun gain**.

Ce résultat est cohérent avec l'EDA : le métier explique 67 % de la variation des salaires, et les autres variables presque 0 %. **Dans ce dataset, le salaire d'une offre se prédit uniquement à partir de son métier**. Les 33 % restants de variation ne sont expliqués par aucune variable disponible : ils ressemblent à un bruit aléatoire autour de la moyenne de chaque métier.

## Portée de l'expérience

La date n'est pas exploitable pour un test temporel : ce découpage ne mesure donc pas la capacité à prédire les salaires futurs. Une erreur moyenne d'environ 18 000 USD reste importante pour une estimation individuelle. Le système de recommandation reste un objectif distinct du projet.

In [4]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Vérifier les variables disponibles dans le notebook
print("Variables disponibles :")
print([v for v in ["y_test", "predictions", "scores"] if v in globals()])


Variables disponibles :
['predictions', 'scores']


In [5]:

print("Type de scores :", type(scores))
print("\nContenu de scores :")
display(scores)


Type de scores : <class 'pandas.core.frame.DataFrame'>

Contenu de scores :


,MAE_USD,RMSE_USD,R2,n_train,n_test
modele,,,,,
Reference moyenne,31452.331979,36998.195609,-4.332320e-07,44000,11000
Moyenne par metier,17870.603779,21318.456640,6.679904e-01,44000,11000
Regression Ridge,17872.053095,21325.896664,6.677586e-01,44000,11000


In [6]:
print("Type de scores :", type(scores))
print("\nContenu de scores :")
display(scores)

Type de scores : <class 'pandas.core.frame.DataFrame'>

Contenu de scores :


,MAE_USD,RMSE_USD,R2,n_train,n_test
modele,,,,,
Reference moyenne,31452.331979,36998.195609,-4.332320e-07,44000,11000
Moyenne par metier,17870.603779,21318.456640,6.679904e-01,44000,11000
Regression Ridge,17872.053095,21325.896664,6.677586e-01,44000,11000


In [7]:
print("Colonnes de scores :", scores.columns.tolist())
print("\nIndex de scores :", scores.index.tolist())
print("\nDimensions :", scores.shape)


Colonnes de scores : ['MAE_USD', 'RMSE_USD', 'R2', 'n_train', 'n_test']

Index de scores : ['Reference moyenne', 'Moyenne par metier', 'Regression Ridge']

Dimensions : (3, 5)


In [8]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Vérifier les variables disponibles dans le notebook
print("Variables disponibles :")
print([v for v in ["y_test", "predictions", "scores"] if v in globals()])

print("Type de scores :", type(scores))
print("\nContenu de scores :")
display(scores)
print("Type de scores :", type(scores))
print("\nContenu de scores :")
display(scores)
print("Colonnes de scores :", scores.columns.tolist())
print("\nIndex de scores :", scores.index.tolist())
print("\nDimensions :", scores.shape)


Variables disponibles :
['predictions', 'scores']
Type de scores : <class 'pandas.core.frame.DataFrame'>

Contenu de scores :


,MAE_USD,RMSE_USD,R2,n_train,n_test
modele,,,,,
Reference moyenne,31452.331979,36998.195609,-4.332320e-07,44000,11000
Moyenne par metier,17870.603779,21318.456640,6.679904e-01,44000,11000
Regression Ridge,17872.053095,21325.896664,6.677586e-01,44000,11000


Type de scores : <class 'pandas.core.frame.DataFrame'>

Contenu de scores :


,MAE_USD,RMSE_USD,R2,n_train,n_test
modele,,,,,
Reference moyenne,31452.331979,36998.195609,-4.332320e-07,44000,11000
Moyenne par metier,17870.603779,21318.456640,6.679904e-01,44000,11000
Regression Ridge,17872.053095,21325.896664,6.677586e-01,44000,11000


Colonnes de scores : ['MAE_USD', 'RMSE_USD', 'R2', 'n_train', 'n_test']

Index de scores : ['Reference moyenne', 'Moyenne par metier', 'Regression Ridge']

Dimensions : (3, 5)


In [9]:
from pathlib import Path

print("Dossier courant :", Path.cwd())
print("Variable project_root définie :", "project_root" in globals())


Dossier courant : c:\wamp64\www\JobMatch\notebooks
Variable project_root définie : True


In [10]:
print("Racine du projet :", project_root)
print("Type :", type(project_root))
print("Dossier data/processed :", project_root / "data" / "processed")
print("Ce dossier existe :", (project_root / "data" / "processed").exists())


Racine du projet : c:\wamp64\www\JobMatch
Type : <class 'pathlib._local.WindowsPath'>
Dossier data/processed : c:\wamp64\www\JobMatch\data\processed
Ce dossier existe : True


In [11]:
output = project_root / "data" / "processed"
output.mkdir(parents=True, exist_ok=True)

fichier_scores = output / "salary_model_metrics.csv"
scores.to_csv(fichier_scores, index=True, index_label="modele")

print("Résultats enregistrés dans :", fichier_scores)
print("Fichier créé :", fichier_scores.exists())

Résultats enregistrés dans : c:\wamp64\www\JobMatch\data\processed\salary_model_metrics.csv
Fichier créé : True
